In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

In [ ]:
class MyMultiheadAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.0, batch_first=True, bias=True):
        super().__init__()
        assert dim % heads == 0, "dim must be divisible by heads"
        self.dim = dim
        self.heads = heads
        self.dh = dim // heads
        self.batch_first = batch_first

        self.to_qkv = nn.Linear(dim, dim * 3, bias=bias)
        self.out_proj = nn.Linear(dim, dim, bias=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        if not self.batch_first:
            x = x.transpose(0, 1)  # (L, B, C) -> (B, L, C)

        B, L, C = x.shape
        q, k, v = self.to_qkv(x).chunk(3, dim=-1)     # (B, L, C) * 3

        # multi head：(B, L, C) -> (B, L, H, Dh) -> (B, H, L, Dh)
        def split_heads(t):
            return t.view(B, L, self.heads, self.dh).transpose(1, 2)
        q = split_heads(q)
        k = split_heads(k)
        v = split_heads(v)

        # scale + softmax
        # attention: each head calculate attn on each position of tokens
        scale = self.dh ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, H, L, L) 
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        y = torch.matmul(attn, v)                            # (B, H, L, Dh)

        # projection to out
        y = y.transpose(1, 2).contiguous().view(B, L, C)
        y = self.out_proj(y)

        if not self.batch_first:
            y = y.transpose(0, 1)
        return y

In [ ]:
# initiate dim, head 
dim, heads = 64, 8
B, L = 2, 7

ref = nn.MultiheadAttention(embed_dim=dim, num_heads=heads,
                            dropout=0.0, bias=True, batch_first=True)
mine = MyMultiheadAttention(dim=dim, heads=heads, dropout=0.0,
                            batch_first=True, bias=True)

# 官方 MHA 的參數化：
# - in_proj_weight: (3*dim, dim)  [Q;K;V] 串在一起
# - in_proj_bias  : (3*dim,)
# - out_proj.{weight,bias}
with torch.no_grad():
    mine.to_qkv.weight.copy_(ref.in_proj_weight)
    mine.to_qkv.bias.copy_(ref.in_proj_bias)
    
    mine.out_proj.weight.copy_(ref.out_proj.weight)
    mine.out_proj.bias.copy_(ref.out_proj.bias)

# --------- 前向與反向數值比對（無 dropout、eval 模式）---------
x = torch.randn(B, L, dim)

ref.eval(); mine.eval()
y_ref, _ = ref(x, x, x, need_weights=False)  # 自注意力：q=k=v=x
y_mine = mine(x)

print("max abs diff (forward):", (y_ref - y_mine).abs().max().item())
assert torch.allclose(y_ref, y_mine, atol=1e-6, rtol=1e-6)

# 也可以測試 backward（把 dropout 關掉且 train()）
ref.train(); mine.train()
x1 = x.clone().requires_grad_(True)
x2 = x.clone().requires_grad_(True)

y1, _ = ref(x1, x1, x1, need_weights=False)
loss1 = (y1 ** 2).mean()
loss1.backward()

y2 = mine(x2)
loss2 = (y2 ** 2).mean()
loss2.backward()

print("max abs diff (input grad):", (x1.grad - x2.grad).abs().max().item())

def max_param_grad_diff(m1, m2):
    diffs = []
    mapping = {
        "to_qkv.weight": "in_proj_weight",
        "to_qkv.bias":   "in_proj_bias",
        "out_proj.weight": "out_proj.weight",
        "out_proj.bias":   "out_proj.bias",
    }
    grads1 = {n: p.grad for n, p in m1.named_parameters() if p.grad is not None}
    grads2 = {n: p.grad for n, p in m2.named_parameters() if p.grad is not None}
    for n1, n2 in mapping.items():
        if n1 in grads1 and n2 in grads2:
            diffs.append((grads1[n1] - grads2[n2]).abs().max().item())
    return max(diffs) if diffs else 0.0

print("max abs diff (param grad):", max_param_grad_diff(mine, ref))


max abs diff (forward): 1.1920928955078125e-07
max abs diff (input grad): 2.3283064365386963e-10
max abs diff (param grad): 1.862645149230957e-09
